# Set Up

In [1]:
# Imports and Helper Functions
import pandas as pd
import RNA
import os
import numpy as np

ModuleNotFoundError: No module named 'RNA'

In [ ]:
def dotbracket_to_pairset(db_string: str) -> set:
    """Converts a dot-bracket string to a set of 1-indexed base pairs."""
    pairs = set()
    stack = []
    for i, char in enumerate(db_string):
        # Use 1-based indexing for biological convention
        pos = i + 1
        if char == '(':
            stack.append(pos)
        elif char == ')':
            if stack:
                j = stack.pop()
                # Store pair as (smaller_index, larger_index)
                pairs.add((j, pos))
    return pairs

def parse_target_region(region_string) -> list:
    """
    Parses a complex region string like '50', '50-52', or '56-63, 69-81'
    into a list of all integer positions.
    """
    if pd.isna(region_string) or region_string == '':
        return []
    region_string = str(region_string).replace(' ', '')
    all_positions = []
    parts = region_string.split(',')
    for part in parts:
        try:
            if '-' in part:
                start, end = map(int, part.split('-'))
                all_positions.extend(range(start, end + 1))
            else:
                all_positions.append(int(part))
        except ValueError:
            print(f"Warning: Could not parse part of a region string: '{part}'")
            continue
    return all_positions

def set_to_str(s: set) -> str:
    """Converts a set of tuples to a sorted, human-readable string for CSV output."""
    if not s:
        return ""
    return "; ".join(map(str, sorted(list(s))))


In [ ]:
# The Core Analysis Engine (Reworked with Correct Order of Operations)

def run_analysis_engine(
    mutant_ids: pd.Series,
    parent_seqs: pd.Series,
    mutant_seqs: pd.Series,
    parent_dbs: pd.Series,
    mutant_dbs: pd.Series,
    goal_types: pd.Series,
    target_regions_1: pd.Series
) -> pd.DataFrame:
    """Analyzes structural differences between parent and mutant RNAs.

    This robust function takes pandas Series of sequence/structure data, calculates
    distance metrics, and performs goal-specific analysis via substring matching.

    Args:
        mutant_ids (pd.Series): Unique string identifiers for each mutant.
        parent_seqs (pd.Series): Parent RNA sequences.
        mutant_seqs (pd.Series): Mutant RNA sequences.
        parent_dbs (pd.Series): Parent dot-bracket structure strings.
        mutant_dbs (pd.Series): Mutant dot-bracket structure strings.
        goal_types (pd.Series): Goal descriptions for substring matching. Keywords:
            "break_pair", "disruption", "extend_stem", "seed_hairpin",
            "compensatory", "stabilizing", "destabilizing".
        target_regions_1 (pd.Series): Primary target region, e.g., '50' or '50-52'.

    Returns:
        pd.DataFrame: A new DataFrame with analysis results, one row per mutant.
    """
    results = []
    
    num_entries = len(mutant_ids)
    print(f"Starting analysis of {num_entries} entries...")

    iterator = zip(
        mutant_ids, parent_seqs, mutant_seqs, parent_dbs, mutant_dbs, 
        goal_types, target_regions_1
    )

    for i, (mutant_id, p_seq, m_seq, p_db, m_db, goal, tr1_str) in enumerate(iterator):
        
        result_row = {'mutant_id': mutant_id, 'goal_type': goal, 'parent_seq': p_seq, 'mutant_seq': m_seq}

        target_region_1 = parse_target_region(tr1_str)
        # --- Core Metric Calculations ---
        
        parent_tree_repr = RNA.db_to_tree_string(p_db, RNA.STRUCTURE_TREE_SHAPIRO_WEIGHT)
        mutant_tree_repr = RNA.db_to_tree_string(m_db, RNA.STRUCTURE_TREE_SHAPIRO_WEIGHT)
        result_row['parent_tree_repr'] = parent_tree_repr
        result_row['mutant_tree_repr'] = mutant_tree_repr
        
        # Calculate Tree Edit Distance 
        tree_parent, tree_mutant = None, None
        try:
            # Create computational Tree objects 
            tree_parent = RNA.make_tree(parent_tree_repr)
            tree_mutant = RNA.make_tree(mutant_tree_repr)
            
            # Calculate distance using the valid tree objects.
            result_row['TreeEditDistance'] = RNA.tree_edit_distance(tree_parent, tree_mutant)
        finally:
            # Free the allocated memory.
            if tree_parent: RNA.free_tree(tree_parent)
            if tree_mutant: RNA.free_tree(tree_mutant)

        # Calculate Base Pair distance
        result_row['Full_BP_Distance'] = RNA.bp_distance(p_db, m_db)
        
        # --- Sequence & Pair Analysis ---
        mutations = [f"{p_seq[i]}{i+1}{m_seq[i]}" for i in range(len(p_seq)) if p_seq[i] != m_seq[i]]
        result_row['Sequence_Changes'] = "; ".join(mutations)
        
        p_parent_set = dotbracket_to_pairset(p_db)
        p_mutant_set = dotbracket_to_pairset(m_db)
        sym_diff = p_parent_set.symmetric_difference(p_mutant_set)
        
        changed_pairs_annotated = []
        for pair_i, pair_j in sorted(list(sym_diff)):
            if (pair_i, pair_j) in p_parent_set:
                pair_str = f"{p_seq[pair_i-1]}{pair_i}-{p_seq[pair_j-1]}{pair_j}"
                changed_pairs_annotated.append(f"Broke {pair_str}")
            elif (pair_i, pair_j) in p_mutant_set:
                pair_str = f"{m_seq[pair_i-1]}{pair_i}-{m_seq[pair_j-1]}{pair_j}"
                changed_pairs_annotated.append(f"Formed {pair_str}")
        result_row['Annotated_Pair_Changes'] = "; ".join(changed_pairs_annotated)
        
        # --- Goal-Specific Logic ---
        on_target_score, off_target_score, intended_changes, logic_type = "N/A", "N/A", set(), "unknown"
        
        if "break_pair" in goal:
            logic_type = "break_pair"
            on_target_score = sum(1 for pos in target_region_1 if m_db[pos - 1] == '.')
            intended_changes = {p for p in p_parent_set if p[0] in target_region_1 or p[1] in target_region_1}
            off_target_pairs = sym_diff - intended_changes
            off_target_score = len(off_target_pairs)

        elif "extend_stem" in goal or "seed_hairpin" in goal:
            logic_type = "make_pair"
            on_target_score = sum(1 for pos in target_region_1 if m_db[pos - 1] in '()')
            intended_changes = {p for p in p_mutant_set if p[0] in target_region_1 or p[1] in target_region_1}
            off_target_pairs = sym_diff - intended_changes
            off_target_score = len(off_target_pairs)

        elif "total_disruption" in goal:
            logic_type = "disruption"
            paired_in_target = sum(1 for pos in target_region_1 if m_db[pos - 1] != '.')
            on_target_score = paired_in_target
            intended_changes = {p for p in p_parent_set if p[0] in target_region_1 or p[1] in target_region_1}
            off_target_pairs = sym_diff - intended_changes
            off_target_score = len(off_target_pairs)

        elif "total_compensation" in goal:
            logic_type = "compensatory"

            parent_partner_map = {i: j for i, j in p_parent_set}
            parent_partner_map.update({j: i for i, j in p_parent_set})
            mutant_partner_map = {i: j for i, j in p_mutant_set}
            mutant_partner_map.update({j: i for i, j in p_mutant_set})

            restored_pair_count = 0
            for pos in target_region_1:
                original_partner = parent_partner_map.get(pos)
                mutant_partner = mutant_partner_map.get(pos)
                if original_partner is not None:
                    if mutant_partner == original_partner:
                        restored_pair_count += 1
                else:
                    if mutant_partner is None:
                        restored_pair_count += 1

            on_target_score = len(target_region_1) - restored_pair_count
            off_target_score = result_row['Full_BP_Distance']

        else:
            print(f"Warning: Goal '{goal}' for mutant '{mutant_id}' did not match any known keywords.")

        result_row['Logic_Type_Applied'] = logic_type
        result_row['OnTargetScore'] = on_target_score
        result_row['OffTargetScore'] = off_target_score
        result_row['IntendedChange_Pairs(positions)'] = set_to_str(intended_changes)
        
        results.append(result_row)
        
    # --- Finalize DataFrame ---
    output_df = pd.DataFrame(results)
    column_order = [
        'mutant_id', 'goal_type', 'Logic_Type_Applied', 'Sequence_Changes',
        'TreeEditDistance', 'OnTargetScore', 'OffTargetScore', 'Full_BP_Distance', 
        'parent_tree_repr', 'mutant_tree_repr', 'Annotated_Pair_Changes', 
        'IntendedChange_Pairs(positions)', "parent_seq", "mutant_seq"
    ]
    # Use reindex to safely order columns, filling missing ones with NaN
    output_df = output_df.reindex(columns=column_order)
    
    print("Analysis function finished.")
    return output_df

print("'run_analysis_engine' function is defined.")

'run_analysis_engine' function is defined.


In [ ]:
# Define PUS7 Union

# Load the two CSV files into pandas DataFrames
incell_pus7_path = "~/PUS7regulation2026/Figure3/plots/PUS7_dep_union_significant_summary.tsv"
invitro_pus7_path = "~/PUS7regulation2026/Figure3/plots/invitro_modified_sites_summary.tsv"

incell_PUS7_df = pd.read_tsv(incell_pus7_path)
invitro_PUS7_df = pd.read_tsv(invitro_pus7_path)

PUS7_union_df = pd.merge(incell_PUS7_df, invitro_PUS7_df, on="chr", how="outer")

print(f"PUS7 data loaded and merged successfully. Number of sequences: {len(PUS7_union_df)}")


pool_path = "~/PUS7regulation2026/Figure4/sequence/pool1_background.csv"
pool_df = pd.read_csv(pool_path)

regex_pattern = r"^T[ACGT]TA[AG]$"
unuar_mask = pool_df['psipos_5mer'].str.contains(regex_pattern, regex=True, na=False)
unuar_df = pool_df.loc[unuar_mask]

print(f"UNUAR data loaded and merged successfully. Number of sequences: {len(unuar_df)}")

PUS7 data loaded and merged successfully. Number of sequences: 296
UNUAR data loaded and merged successfully. Number of sequences: 337


# Execution  

## Psi Pairing

In [ ]:
pool_access_psi="~/PUS7regulation2026/Figure5/mutagenesis_evaluation/pairing_status_flip_analysis_pool1.csv"

pool_access_psi_df = pd.read_csv(pool_access_psi)

column_mapping = {
    'mutant_ids': pool_access_psi_df['name'],
    'parent_seqs': pool_access_psi_df['original_seq'],
    'mutant_seqs': pool_access_psi_df['mutated_seq'],
    'parent_dbs': pool_access_psi_df['original_dot_bracket'],
    'mutant_dbs': pool_access_psi_df['dot_bracket_mut'],
    'goal_types': pool_access_psi_df['strategy'],
    'target_regions_1': pool_access_psi_df['target_pos']
}

print("DataFrame loaded successfully.")

results_df = run_analysis_engine(**column_mapping)

dfs_by_logic = {}

for name, group in results_df.groupby('Logic_Type_Applied'):
    print(f"Creating DataFrame for logic type: '{name}' with {len(group)} rows.")
    dfs_by_logic[name] = group

pool_break_pair_df = dfs_by_logic.get('break_pair')
pool_make_pair_df = dfs_by_logic.get('make_pair')

poolPUS7_make_pair_df = pool_make_pair_df[pool_make_pair_df['mutant_id'].isin(PUS7_union_df['chr'])]
poolPUS7_break_pair_df = pool_break_pair_df[pool_break_pair_df['mutant_id'].isin(unuar_df['chr'])]

print(f"Number of break pair: {len(poolPUS7_break_pair_df)}")
print(f"Number of make pair: {len(poolPUS7_make_pair_df)}")

pool_break_pair_ted_threshold = poolPUS7_break_pair_df['TreeEditDistance'].quantile(0.5)
pool_make_pair_ted_threshold = poolPUS7_make_pair_df['TreeEditDistance'].quantile(0.5)

pool_break_pair_off_threshold = poolPUS7_break_pair_df['OffTargetScore'].quantile(0.5)
pool_make_pair_off_threshold = poolPUS7_make_pair_df['OffTargetScore'].quantile(0.5)

pool_good_break_pair_df = poolPUS7_break_pair_df.loc[(poolPUS7_break_pair_df['OnTargetScore'] == 1)
                                        & (poolPUS7_break_pair_df['TreeEditDistance'] <= pool_break_pair_ted_threshold) 
                                        & (poolPUS7_break_pair_df['OffTargetScore'] <= pool_break_pair_off_threshold)
                                        ]
pool_good_make_pair_df = poolPUS7_make_pair_df.loc[(poolPUS7_make_pair_df['OnTargetScore'] == 1)
                                        &  (poolPUS7_make_pair_df['TreeEditDistance'] <= pool_make_pair_ted_threshold) 
                                        & (poolPUS7_make_pair_df['OffTargetScore'] <= pool_make_pair_off_threshold)
                                        ]

print(f"Number of good break pair: {len(pool_good_break_pair_df)}")
print(f"Number of good make pair: {len(pool_good_make_pair_df)}")

pool_good_break_pair_df.to_csv("pool_good_breakpair_psi.csv", index=False)
pool_good_make_pair_df.to_csv("pool_good_makepair_psi.csv", index=False)
print(f"\nAnalysis complete. Results saved.")


DataFrame loaded successfully.
Starting analysis of 634 entries...
Analysis function finished.
Creating DataFrame for logic type: 'break_pair' with 234 rows.
Creating DataFrame for logic type: 'make_pair' with 400 rows.
Number of break pair: 112
Number of make pair: 150
Number of good break pair: 31
Number of good make pair: 39

Analysis complete. Results saved.


## Psi +1 Pairing Accessibility

In [ ]:
pool_access_psi="~/PUS7regulation2026/Figure5/mutagenesis_evaluation/pairing_status_flip_analysis01_pool1.csv"

pool_access_psi_df = pd.read_csv(pool_access_psi)

column_mapping = {
    'mutant_ids': pool_access_psi_df['name'],
    'parent_seqs': pool_access_psi_df['original_seq'],
    'mutant_seqs': pool_access_psi_df['mutated_seq'],
    'parent_dbs': pool_access_psi_df['original_dot_bracket'],
    'mutant_dbs': pool_access_psi_df['dot_bracket_mut'],
    'goal_types': pool_access_psi_df['strategy'],
    'target_regions_1': pool_access_psi_df['target_pos']
}

print("DataFrame loaded successfully.")

results_df = run_analysis_engine(**column_mapping)

dfs_by_logic = {}

for name, group in results_df.groupby('Logic_Type_Applied'):
    print(f"Creating DataFrame for logic type: '{name}' with {len(group)} rows.")
    dfs_by_logic[name] = group

pool_break_pair_df = dfs_by_logic.get('break_pair')
pool_make_pair_df = dfs_by_logic.get('make_pair')

poolPUS7_make_pair_df = pool_make_pair_df[pool_make_pair_df['mutant_id'].isin(PUS7_union_df['chr'])]
poolPUS7_break_pair_df = pool_break_pair_df[pool_break_pair_df['mutant_id'].isin(unuar_df['chr'])]

print(f"Number of break pair: {len(poolPUS7_break_pair_df)}")
print(f"Number of make pair: {len(poolPUS7_make_pair_df)}")

pool_break_pair_ted_threshold = poolPUS7_break_pair_df['TreeEditDistance'].quantile(0.5)
pool_make_pair_ted_threshold = poolPUS7_make_pair_df['TreeEditDistance'].quantile(0.5)

pool_break_pair_off_threshold = poolPUS7_break_pair_df['OffTargetScore'].quantile(0.5)
pool_make_pair_off_threshold = poolPUS7_make_pair_df['OffTargetScore'].quantile(0.5)

pool_good_break_pair_df = poolPUS7_break_pair_df.loc[(poolPUS7_break_pair_df['OnTargetScore'] == 2)
                                        & (poolPUS7_break_pair_df['TreeEditDistance'] <= pool_break_pair_ted_threshold) 
                                        & (poolPUS7_break_pair_df['OffTargetScore'] <= pool_break_pair_off_threshold)
                                        ]
pool_good_make_pair_df = poolPUS7_make_pair_df.loc[(poolPUS7_make_pair_df['OnTargetScore'] == 2)
                                        &  (poolPUS7_make_pair_df['TreeEditDistance'] <= pool_make_pair_ted_threshold) 
                                        & (poolPUS7_make_pair_df['OffTargetScore'] <= pool_make_pair_off_threshold)
                                        ]

print(f"Number of good break pair: {len(pool_good_break_pair_df)}")
print(f"Number of good make pair: {len(pool_good_make_pair_df)}")

pool_good_break_pair_df.to_csv("pool_good_breakpair_psi01.csv", index=False)
pool_good_make_pair_df.to_csv("pool_good_makepair_psi01.csv", index=False)
print(f"\nAnalysis complete. Results saved.")


DataFrame loaded successfully.
Starting analysis of 644 entries...
Analysis function finished.
Creating DataFrame for logic type: 'break_pair' with 306 rows.
Creating DataFrame for logic type: 'make_pair' with 338 rows.
Number of break pair: 148
Number of make pair: 126
Number of good break pair: 25
Number of good make pair: 35

Analysis complete. Results saved.


## Psipos 5mer Pairing Accessibility

In [ ]:
pool_access_psi="~/PUS7regulation2026/Figure5/mutagenesis_evaluation/pairing_status_flip_analysis5mer_pool1.csv"

pool_access_psi_df = pd.read_csv(pool_access_psi)

column_mapping = {
    'mutant_ids': pool_access_psi_df['name'],
    'parent_seqs': pool_access_psi_df['original_seq'],
    'mutant_seqs': pool_access_psi_df['mutated_seq'],
    'parent_dbs': pool_access_psi_df['original_dot_bracket'],
    'mutant_dbs': pool_access_psi_df['dot_bracket_mut'],
    'goal_types': pool_access_psi_df['strategy'],
    'target_regions_1': pool_access_psi_df['target_pos_range']
}

print("DataFrame loaded successfully.")

results_df = run_analysis_engine(**column_mapping)

dfs_by_logic = {}

for name, group in results_df.groupby('Logic_Type_Applied'):
    print(f"Creating DataFrame for logic type: '{name}' with {len(group)} rows.")
    dfs_by_logic[name] = group

pool_break_pair_df = dfs_by_logic.get('break_pair')
pool_make_pair_df = dfs_by_logic.get('make_pair')

poolPUS7_make_pair_df = pool_make_pair_df[pool_make_pair_df['mutant_id'].isin(PUS7_union_df['chr'])]
poolPUS7_break_pair_df = pool_break_pair_df[pool_break_pair_df['mutant_id'].isin(unuar_df['chr'])]

print(f"Number of break pair: {len(poolPUS7_break_pair_df)}")
print(f"Number of make pair: {len(poolPUS7_make_pair_df)}")

pool_break_pair_ted_threshold = poolPUS7_break_pair_df['TreeEditDistance'].quantile(0.5)
pool_make_pair_ted_threshold = poolPUS7_make_pair_df['TreeEditDistance'].quantile(0.5)

pool_break_pair_off_threshold = poolPUS7_break_pair_df['OffTargetScore'].quantile(0.5)
pool_make_pair_off_threshold = poolPUS7_make_pair_df['OffTargetScore'].quantile(0.5)

pool_good_break_pair_df = poolPUS7_break_pair_df.loc[(poolPUS7_break_pair_df['OnTargetScore'] == 5)
                                        & (poolPUS7_break_pair_df['TreeEditDistance'] <= pool_break_pair_ted_threshold) 
                                        & (poolPUS7_break_pair_df['OffTargetScore'] <= pool_break_pair_off_threshold)
                                        ]
pool_good_make_pair_df = poolPUS7_make_pair_df.loc[(poolPUS7_make_pair_df['OnTargetScore'] == 5)
                                        &  (poolPUS7_make_pair_df['TreeEditDistance'] <= pool_make_pair_ted_threshold) 
                                        & (poolPUS7_make_pair_df['OffTargetScore'] <= pool_make_pair_off_threshold)
                                        ]

print(f"Number of good break pair: {len(pool_good_break_pair_df)}")
print(f"Number of good make pair: {len(pool_good_make_pair_df)}")

pool_good_break_pair_df.to_csv("pool_good_breakpair_5mer.csv", index=False)
pool_good_make_pair_df.to_csv("pool_good_makepair_5mer.csv", index=False)
print(f"\nAnalysis complete. Results saved.")


DataFrame loaded successfully.
Starting analysis of 628 entries...
Analysis function finished.
Creating DataFrame for logic type: 'break_pair' with 495 rows.
Creating DataFrame for logic type: 'make_pair' with 133 rows.
Number of break pair: 247
Number of make pair: 50
Number of good break pair: 28
Number of good make pair: 7

Analysis complete. Results saved.


## Unpaired2 Accessibility

In [ ]:
pool_access="~/PUS7regulation2026/Figure5/mutagenesis_evaluation/pairing_status_flip_analysis_pool1_unpaired2.csv"

pool_access_df = pd.read_csv(pool_access)

column_mapping = {
    'mutant_ids': pool_access_df['name'],
    'parent_seqs': pool_access_df['original_seq'],
    'mutant_seqs': pool_access_df['mutated_seq'],
    'parent_dbs': pool_access_df['original_dot_bracket'],
    'mutant_dbs': pool_access_df['dot_bracket_mut'],
    'goal_types': pool_access_df['strategy'],
    'target_regions_1': pool_access_df['target_pos_range']
}

print("DataFrame loaded successfully.")

results_df = run_analysis_engine(**column_mapping)

dfs_by_logic = {}

for name, group in results_df.groupby('Logic_Type_Applied'):
    print(f"Creating DataFrame for logic type: '{name}' with {len(group)} rows.")
    dfs_by_logic[name] = group

pool_break_pair_df = dfs_by_logic.get('break_pair')
pool_make_pair_df = dfs_by_logic.get('make_pair')

# The concise, single-line solution
poolPUS7_make_pair_df = pool_make_pair_df[pool_make_pair_df['mutant_id'].isin(PUS7_union_df['chr'])]
poolPUS7_break_pair_df = pool_break_pair_df[pool_break_pair_df['mutant_id'].isin(unuar_df['chr'])]

print(f"Number of break pair: {len(poolPUS7_break_pair_df)}")
print(f"Number of make pair: {len(poolPUS7_make_pair_df)}")

pool_break_pair_ted_threshold = poolPUS7_break_pair_df['TreeEditDistance'].quantile(0.5)
pool_make_pair_ted_threshold = poolPUS7_make_pair_df['TreeEditDistance'].quantile(0.5)

pool_break_pair_off_threshold = poolPUS7_break_pair_df['OffTargetScore'].quantile(0.5)
pool_make_pair_off_threshold = poolPUS7_make_pair_df['OffTargetScore'].quantile(0.5)

pool_good_break_pair_df = poolPUS7_break_pair_df.loc[(poolPUS7_break_pair_df['OnTargetScore'] == 3)
                                        & (poolPUS7_break_pair_df['TreeEditDistance'] <= pool_break_pair_ted_threshold) 
                                        & (poolPUS7_break_pair_df['OffTargetScore'] <= pool_break_pair_off_threshold)
                                        ]
pool_good_make_pair_df = poolPUS7_make_pair_df.loc[(poolPUS7_make_pair_df['OnTargetScore'] == 3)
                                        &  (poolPUS7_make_pair_df['TreeEditDistance'] <= pool_make_pair_ted_threshold) 
                                        & (poolPUS7_make_pair_df['OffTargetScore'] <= pool_make_pair_off_threshold)
                                        ]

print(f"Number of good break pair: {len(pool_good_break_pair_df)}")
print(f"Number of good make pair: {len(pool_good_make_pair_df)}")

pool_good_break_pair_df.to_csv("pool_good_breakpair_unpaired2.csv", index=False)
pool_good_make_pair_df.to_csv("pool_good_makepair_unpaired2.csv", index=False)
print(f"\nAnalysis complete. Results saved.")


DataFrame loaded successfully.
Starting analysis of 604 entries...
Analysis function finished.
Creating DataFrame for logic type: 'break_pair' with 419 rows.
Creating DataFrame for logic type: 'make_pair' with 185 rows.
Number of break pair: 196
Number of make pair: 84
Number of good break pair: 33
Number of good make pair: 17

Analysis complete. Results saved.


## Structure Mutation

In [ ]:
pool_structure="~/PUS7regulation2026/Figure5/mutagenesis_evaluation/all_mutagenesis_results.csv"

pool_structure_df = pd.read_csv(pool_structure)

# filter for PUS7 Union
pool_structure_df['name'] = pool_structure_df['original_filename'].str.removesuffix('.fold')
poolPUS7_structure_df = pool_structure_df[pool_structure_df['name'].isin(PUS7_union_df['chr'])]

print(f"Number of mutations in PUS7 union: {len(poolPUS7_structure_df)}")

# Remove duplicated total seqeunces that match an up or downstream mutant
priority_order = [
    'upstream_disrution',
    'downstream_disruption',
    'total_disruption',
    'upstream_compensation',
    'downstream_compensation',
    'total_compensation'
]

poolPUS7_structure_df['goal_type_cat'] = pd.Categorical(
    poolPUS7_structure_df['goal_type'],
    categories=priority_order,
    ordered=True
)

filtered_poolPUS7_structure_df = poolPUS7_structure_df.sort_values('goal_type_cat').drop_duplicates(
    subset='mutant_seq', 
    keep='first'
)

print(f"Number of mutations in filtered PUS7 union: {len(filtered_poolPUS7_structure_df)}")


column_mapping = {
    'mutant_ids': filtered_poolPUS7_structure_df['mutant_id'],
    'parent_seqs': filtered_poolPUS7_structure_df['parent_seq'],
    'mutant_seqs': filtered_poolPUS7_structure_df['mutant_seq'],
    'parent_dbs': filtered_poolPUS7_structure_df['parent_db'],
    'mutant_dbs': filtered_poolPUS7_structure_df['mutant_db'],
    'goal_types': filtered_poolPUS7_structure_df['goal_type'],
    'target_regions_1': filtered_poolPUS7_structure_df['target_mut_region']
}

print("DataFrame loaded successfully.")

results_df = run_analysis_engine(**column_mapping)

dfs_by_logic = {}

for name, group in results_df.groupby('goal_type'):
    print(f"Creating DataFrame for logic type: '{name}' with {len(group)} rows.")
    dfs_by_logic[name] = group

total_disr_df = dfs_by_logic.get('total_disruption')
total_comp_df = dfs_by_logic.get('total_compensation')

tot_disrupt_on_threshold = total_disr_df['OnTargetScore'].quantile(0.5)
tot_disrupt_ted_threshold = total_disr_df['TreeEditDistance'].quantile(0.5)
tot_disrupt_off_threshold = total_disr_df['OffTargetScore'].quantile(0.3)

good_tot_disr_df = total_disr_df.loc[(total_disr_df['OnTargetScore'] <= tot_disrupt_on_threshold)
                                        & (total_disr_df['TreeEditDistance'] <= tot_disrupt_ted_threshold) 
                                        & (total_disr_df['OffTargetScore'] <= tot_disrupt_off_threshold)
                                        ]
print()
print(f"Number of good total disruptions: {len(good_tot_disr_df)}")


successful_tot_disr = set(
    good_tot_disr_df['mutant_id'].str.replace(r'_disruption$|_compensation$', '', regex=True)
)
final_tot_comp_df = total_comp_df[
    total_comp_df['mutant_id'].str.replace(r'_disruption$|_compensation$', '', regex=True).isin(successful_tot_disr)
].copy()

print()
print(f"Number of total compensations: {len(final_tot_comp_df)}")

tot_comp_on_threshold = final_tot_comp_df['OnTargetScore'].quantile(0.5)
tot_comp_ted_threshold = final_tot_comp_df['TreeEditDistance'].quantile(0.5)
tot_comp_off_threshold = final_tot_comp_df['OffTargetScore'].quantile(0.5)

good_tot_comp_df = final_tot_comp_df.loc[(final_tot_comp_df['OnTargetScore'] <= tot_comp_on_threshold)
                                        & (final_tot_comp_df['TreeEditDistance'] <= tot_comp_ted_threshold) 
                                        & (final_tot_comp_df['OffTargetScore'] <= tot_comp_off_threshold)
                                        ]
print()
print(f"Number of good total compensations: {len(good_tot_comp_df)}")

# Specific sites to include
sites = ["RHBDD2", "IGF2BP1", "FIS1", "HDAC6", "EIF5"]
specsite_structure_df = results_df[results_df['mutant_id'].str.contains('|'.join(sites), case=False, na=False)]

specsite_structure_df.to_csv("pool_specific_struct_mut.csv", index=False)
good_tot_disr_df.to_csv("pool_good_tot_disruption.csv", index=False)
good_tot_comp_df.to_csv("pool_good_tot_compensation.csv", index=False)
print(f"\nAnalysis complete. Results saved.")


Number of mutations in PUS7 union: 1516
Number of mutations in filtered PUS7 union: 1343
DataFrame loaded successfully.
Starting analysis of 1343 entries...


C:\Users\rmrex\AppData\Local\Temp\ipykernel_20396\1465844968.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  poolPUS7_structure_df['goal_type_cat'] = pd.Categorical(


Analysis function finished.
Creating DataFrame for logic type: 'downstream_compensation' with 239 rows.
Creating DataFrame for logic type: 'downstream_disruption' with 260 rows.
Creating DataFrame for logic type: 'total_compensation' with 184 rows.
Creating DataFrame for logic type: 'total_disruption' with 225 rows.
Creating DataFrame for logic type: 'upstream_compensation' with 222 rows.
Creating DataFrame for logic type: 'upstream_disruption' with 213 rows.

Number of good downstream disruptions: 47
Number of good upstream disruptions: 29
Number of good total disruptions: 34

Number of downstream compensations: 42
Number of upstream compensations: 29
Number of total compensations: 26

Number of good downstream compensations: 11
Number of good upstream compensations: 12
Number of good total compensations: 9

Analysis complete. Results saved.
